# Density based clustering methods

In this practice you will:
- implement the clustering method presented in the paper: Rodriguez, Alex, and Alessandro Laio. “Clustering by Fast Search and Find of Density Peaks.” Science 344, no. 6191 (2014): 1492–96. https://doi.org/10.1126/science.1242072.
- apply it to the MNIST dataset
- apply the DBSCAN method on the MNIST dataset
- learn how clustering performances with these two methods are affected by the parameters of the algorithms

## 0. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score

## 1. Create a synthetic dataset

In [ ]:
mean1 = [1, 6]
cov1 = [[0.5, 0.0],
        [0.0, 0.5]]
X1 = np.random.multivariate_normal(mean1, cov1, 120)
mean2 = [9, 7]
cov2 = [[2.5, 1.8],
        [1.8, 1.5]]
X2 = np.random.multivariate_normal(mean2, cov2, 180)
mean3 = [-4, 4]
cov3 = [[4.2, -2.4],
        [-2.4, 4.0]]
X3 = np.random.multivariate_normal(mean3, cov3, 100)
X = np.vstack([X1, X2, X3])
y_true = np.array([0]*len(X1) + [1]*len(X2) + [2]*len(X3))
feature_names = ['feat_1', 'feat_2']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pd.DataFrame(X_scaled, columns=feature_names).describe().loc[["mean", "std"]]
plt.scatter(
    X_scaled[:, 0],
    X_scaled[:, 1],
    s=25,
    alpha=0.7
)
plt.xlabel(feature_names[0] + " (scaled)")
plt.ylabel(feature_names[1] + " (scaled)")
plt.title("Synthetic dataset")
plt.show()

## 2. Clustering based on density peaks

Write a function to calculate for each point in the dataset:
* the local density, **rho**, defined as the number of samples within a given distance epsilon;
* the index of the closest sample with higher density, **inds_closest_higher_rho**. For the sample with highest density set this value to -1;
* the distance to the closest sample with higher density, **delta**.

The function should returns the arrays rho, inds_closest_higher_rho, and delta, and it should plot delta as a function of rho. Test the function using the synthetic dataset, and values of epsilon equal to 0.1, 1, and 10. How does the plot change ? What is the optimal value ?

Write a function that takes as inputs the arrays produced by the previous function, and threshold values for rho, **min_rho**, and delta, **min_delta**. The function should:
* define as cluster centers all the points with rho > min_rho and delta > min_delta. I suggest to highlight the cluster centers in a plot of delta Vs rho, to check that the correct samples were selected;
* assign each point to the same cluster of its closest sample of higher density. To do this I suggest to:
  * assign a cluster label to the cluster centers
  * make a cycle over samples in decreasing order of density
      * if the sample does not have a cluster label already assigned to it, use the cluster label of its closest sample of higher density

The function should return the labels of the clusters. Apply the function to the synthetic data, and plot the data using different colors for the clusters. Define the values of min_rho and min_delta based on the plot produced by the previous function.

## 3. Clustering of the MNIST dataset based on density peaks

The MNIST dataset includes images of hand-written digits. Each image is 8x8. The block of code below load the data, **X**, the correct labels, **y**, and it plots a random image from the dataset.

* Use the functions defined before to identify possible candidates as cluster centers.Modify the value of epsilon to get a number of isolated density peaks around 10-15
* Perform the clustering
* Compute the adjusted rank index of the clustering results
* Plots some examples of images for all the clusters identified

In [ ]:
images = datasets.load_digits()
X = images['data']
n_images = X.shape[0]
y = images['target']
print('Number of images:', n_images)
X_images = X.reshape(n_images, 8, 8)
ind = np.random.randint(n_images)
plt.imshow(X_images[ind], cmap='grey')
plt.title(y[ind])

In [ ]:
# this block of code imports a different dataset with images at higher resolution. I suggest to use the previous one,
# and then maybe explore this one. In case remember that you need to adjust the parameters of the algoriths,
# as the average distance between samples would be significanly different for these images
from sklearn.datasets import fetch_openml
n_images = 1000 # I downsample the original dataset to this number
X,y = fetch_openml('mnist_784', return_X_y=True)
inds = np.random.choice(np.arange(len(y)).astype(int), n_images)
X = X.to_numpy()
y = y.to_numpy()
X = X[inds,:]
y = y[inds]
n_images = X.shape[0]
X_images = X.reshape(n_images, 28, 28)
X = X_images.reshape(n_images, 28 * 28)
print('Number of images:', n_images)
ind = np.random.randint(n_images)
plt.imshow(X_images[ind], cmap='grey')
plt.title(y[ind])

## 4. Clustering of the MNIST dataset using DBSCAN

Clusterize the same data using the DBSCAN algorithm. Set epsilon to 20 and the minimum number of samples to 10. Compute the adjusted rank index excluding the samples labelled ad noise by DBSCAN.